<p align="center">
  <br/>
    <a href="https://stabrise.com/scaledp/"><img alt="ScaleDP" src="https://raw.githubusercontent.com/StabRise/ScaleDP/refs/heads/master/images/scaledp.webp" width="376" style="max-width: 100%;"></a>
  <br/>
</p>

<p align="center">
    <a target="_blank" href="https://colab.research.google.com/github/StabRise/scaledp-tutorials/blob/master/embeddings/1.TextSplitterAndEmbeddings.ipynb">
        <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
    </a>
    <a href="https://github.com/stabrise/scaledp/" alt="GitHub"><img src="https://img.shields.io/static/v1?label=GitHub&message=ScaleDP&color=blue&logo=github" /></a>
    <a href="https://pypi.org/project/scaledp/" alt="Package on PyPI"><img src="https://img.shields.io/pypi/v/scaledp.svg" /></a>    
    <a href="https://github.com/stabrise/scaledp/blob/main/LICENSE"><img alt="GitHub" src="https://img.shields.io/github/license/stabrise/scaledp.svg?color=blue"></a>
    <a href="https://stabrise.com"><img alt="StabRise" src="https://img.shields.io/badge/by-StabRise-orange.svg?style=flat&colorA=E1523D&colorB=blue"></a>
    <a href="https://scaledp.stabrise.com/en/latest/"><img src="https://app.readthedocs.org/projects/scaledp/badge/?version=latest" alt="Documentation Status"/></a>
</p>

# Text Splitting and Embeddings with ScaleDP

This tutorial demonstrates the complete workflow for document processing using ScaleDP:
1. **Read PDF** - Load a PDF document
2. **Split Text** - Break the document into semantic chunks using TextSplitter
3. **Generate Embeddings** - Create vector embeddings for each text chunk using TextEmbeddings

This workflow is useful for building semantic search, similarity analysis, and RAG (Retrieval-Augmented Generation) applications.

## Requirements

- Python 3.8+
- Java 8 (for Spark)
- scaledp library with embedding dependencies

[Learn more about TextSplitter](https://scaledp.stabrise.com/en/latest/models/splitters/text_splitter.html)

[Learn more about TextEmbeddings](https://scaledp.stabrise.com/en/latest/models/embeddings/text_embeddings.html)

## Installation and Setup

### Automatically install Java 8 on Google Colab

In [ ]:
%%bash
if python -c 'import google.colab' &> /dev/null; then
    echo "Running on Google Colab!"
    echo "Installing OpenJDK 8..."
    apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
    java -version
fi

### Set JAVA_HOME environment variable

In [ ]:
import os
import sys

if 'google.colab' in sys.modules:
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
    print("✓ Java environment configured for Colab")
else:
    print("✓ Running locally with local Java installation")

### Install ScaleDP

Install the scaledp package with embeddings support (includes sentence-transformers).

In [ ]:
!pip install -q scaledp

### Start Spark Session with ScaleDP

In [2]:
from scaledp import *

# Verify Spark
spark = ScaleDPSession()
spark

## Workflow Overview

### 1. PDF to Document
Read a PDF file and convert it to text documents

### 2. Text Splitting
Break documents into semantic chunks using the TextSplitter model, which intelligently splits text while preserving context

### 3. Embeddings Generation
Convert text chunks into dense vector embeddings using a pre-trained language model (default: `all-MiniLM-L6-v2`)

Each step preserves metadata (path, processing time, etc.) for traceability.

## Step 1: Load Sample PDF

Load sample from resources of ScaleDP package.

In [3]:
pdf_example = files('resources/pdfs/example.pdf')

### Load PDF and Build Complete Processing Pipeline

We'll create a single unified ML pipeline that handles all steps:
1. **TextSplitter** - Split document into semantic chunks  
2. **TextEmbeddings** - Generate embeddings for each chunk

Spark's PDF datasource automatically provides a `document` column with extracted text, so we can feed it directly into our pipeline.

In [4]:
## Read PDF file to the DataFrame
# Load PDF using Spark's PDF datasource
# This automatically provides a 'document' column with extracted text
df = spark.read.format("pdf").load(pdf_example)

print("✓ PDF DataFrame loaded with document column")
print(f"  Files: {df.count()}")
print("\nDataFrame Schema:")
df.printSchema()

✓ PDF DataFrame loaded with document column
  Files: 1

DataFrame Schema:
root
 |-- path: string (nullable = true)
 |-- filename: string (nullable = true)
 |-- page_number: integer (nullable = true)
 |-- partition_number: integer (nullable = true)
 |-- text: string (nullable = true)
 |-- image: struct (nullable = true)
 |    |-- path: string (nullable = true)
 |    |-- resolution: integer (nullable = true)
 |    |-- data: binary (nullable = true)
 |    |-- imageType: string (nullable = true)
 |    |-- exception: string (nullable = true)
 |    |-- height: integer (nullable = true)
 |    |-- width: integer (nullable = true)
 |-- document: struct (nullable = true)
 |    |-- path: string (nullable = true)
 |    |-- text: string (nullable = true)
 |    |-- outputType: string (nullable = true)
 |    |-- bBoxes: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- text: string (nullable = true)
 |    |    |    |-- score: float (nullable = true)
 |  

## Step 2: Build Pipeline for Text Splitting and Embeddings

Create a single PipelineModel that combines TextSplitter and TextEmbeddings stages for efficient end-to-end processing.

### Configure Pipeline Stages

Define the two stages of our unified pipeline:

**Stage 1: TextSplitter**
- Splits the Document into semantic chunks
- Uses intelligent splitting that preserves meaning and context
- Parameters:
  - `inputCol`: "document" (from PDF datasource)
  - `outputCol`: "chunks" (TextChunks for next stage)
  - `chunk_size`: Target size of each chunk in characters

**Stage 2: TextEmbeddings**
- Converts each chunk into a vector embedding
- Uses pre-trained sentence-transformers model
- Parameters:
  - `inputCol`: "chunks" (from TextSplitter)
  - `outputCol`: "embeddings" (final output)
  - `model`: Sentence transformer model identifier

In [10]:
# Stage 1: Create TextSplitter
# Adjust chunk_size based on your use case:
# - Smaller chunks (512-768): Better for detailed information retrieval
# - Medium chunks (1024-2048): Balanced for most use cases
# - Larger chunks (2048+): For documents requiring more context

text_splitter = TextSplitter(
    inputCol="document",      # From PDF datasource
    outputCol="chunks",       # TextChunks for embeddings
    chunk_size=512           # Target chunk size in characters
)

# Stage 2: Create TextEmbeddings
# Note: First run will download the model (~22MB for all-MiniLM-L6-v2)

text_embeddings = TextEmbeddings(
    inputCol="chunks",          # From TextSplitter
    outputCol="embeddings",     # Final embeddings output
    model="all-MiniLM-L6-v2",   # Sentence transformer model (384-dimensional)
    keepInputData=True,         # Keep input chunks in output
    batchSize=32,               # Process 32 chunks at a time
    partitionMap=True           # Use partition mapping for efficiency
)


### Create and Execute Pipeline

Combine both stages into a single PipelineModel and apply it to the PDF data.

In [11]:
# Create unified pipeline with both stages
pipeline = PipelineModel(stages=[text_splitter, text_embeddings])

# Execute the pipeline: PDF document → chunks → embeddings
print("\nExecuting pipeline...")
df_results = pipeline.transform(df).cache()

print("✓ Pipeline execution complete")
print("\nOutput DataFrame Schema:")
df_results.printSchema()


Executing pipeline...
✓ Pipeline execution complete

Output DataFrame Schema:
root
 |-- chunks: struct (nullable = true)
 |    |-- path: string (nullable = true)
 |    |-- page: integer (nullable = true)
 |    |-- chunks: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- exception: string (nullable = true)
 |    |-- processing_time: double (nullable = true)
 |-- embeddings: struct (nullable = true)
 |    |-- path: string (nullable = true)
 |    |-- page: integer (nullable = true)
 |    |-- text: string (nullable = true)
 |    |-- data: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- type: string (nullable = true)
 |    |-- exception: string (nullable = true)
 |    |-- processing_time: double (nullable = true)



### Examine Output Results

Display sample chunks and embeddings from the pipeline output.

In [13]:
# Display sample chunks using show_text utility
df_results.show()

+--------------------+--------------------+
|              chunks|          embeddings|
+--------------------+--------------------+
|{file:///home/myk...|{file:///home/myk...|
|{file:///home/myk...|{file:///home/myk...|
|{file:///home/myk...|{file:///home/myk...|
+--------------------+--------------------+



In [14]:
df_results.select("embeddings.*").show(truncate=20)

+--------------------+----+--------------------+--------------------+----------+---------+--------------------+
|                path|page|                text|                data|      type|exception|     processing_time|
+--------------------+----+--------------------+--------------------+----------+---------+--------------------+
|file:///home/myko...|   0|On October 21, 20...|[0.05693748593330...|text_chunk|         |0.021975199381510418|
|file:///home/myko...|   0|Meanwhile, Micros...|[-0.0254266485571...|text_chunk|         |0.021975199381510418|
|file:///home/myko...|   0|The impact of thi...|[0.00468476628884...|text_chunk|         |0.021975199381510418|
+--------------------+----+--------------------+--------------------+----------+---------+--------------------+



## Step 3: Analyze Pipeline Results

Examine the embeddings generated by the unified pipeline.

### Embeddings Output Structure

The `embeddings` column contains `EmbeddingsOutput` objects with the following fields:
- `path`: File path or "memory" for in-memory data
- `page`: Page number if applicable
- `text`: Original text chunk
- `data`: The embedding vector (384 dimensions for all-MiniLM-L6-v2)
- `type`: Type of embedding ("text_chunk" for this case)
- `exception`: Error message if embedding generation failed


The pipeline preserves the `chunks` column so you can correlate embeddings with their source text.

In [15]:
from pyspark.sql.functions import col, size

# Display embeddings metadata
embeddings_info = df_results.select(
    col("embeddings.data").alias("embedding_vector"),
    size(col("embeddings.data")).alias("embedding_dim"),
    col("embeddings.exception").alias("exception")
)

print("Embeddings Information:")
embeddings_info.show(truncate=40)

Embeddings Information:
+----------------------------------------+-------------+---------+
|                        embedding_vector|embedding_dim|exception|
+----------------------------------------+-------------+---------+
|[0.05693748593330383, -0.069871962070...|          384|         |
|[-0.025426648557186127, -0.0391667447...|          384|         |
|[0.004684766288846731, -0.01358991488...|          384|         |
+----------------------------------------+-------------+---------+



## Use Cases

This workflow enables several powerful applications:

### 1. **Semantic Search**
- Convert user queries to embeddings
- Find most similar chunks using cosine similarity
- Rank results by relevance

### 2. **Similarity Clustering**
- Group similar chunks together
- Identify duplicate or near-duplicate content
- Detect topic changes in documents

### 3. **Retrieval-Augmented Generation (RAG)**
- Store embeddings in vector database
- Retrieve relevant chunks for LLM context
- Improve LLM responses with document-specific knowledge

### 4. **Document Categorization**
- Classify documents by embedding similarity
- Organize large document collections
- Route documents to appropriate handlers

### 5. **Anomaly Detection**
- Identify unusual or out-of-context chunks
- Detect structural inconsistencies
- Flag suspicious content

## Next Steps

1. **Store in Vector Database**
   - Save embeddings to Pinecone, Weaviate, or Milvus
   - Enable fast similarity search at scale

2. **Build Semantic Search**
   - Query with natural language
   - Retrieve most relevant chunks
   - Display results with context

3. **Implement RAG Pipeline**
   - Combine with LLMs (GPT, Claude, etc.)
   - Use retrieved chunks as context
   - Generate grounded answers

4. **Scale to Production**
   - Process large document collections
   - Monitor processing metrics
   - Handle errors and edge cases

5. **Fine-tune for Your Domain**
   - Use domain-specific embedding models
   - Adjust chunk sizes for your data
   - Evaluate embedding quality

## Summary

In this tutorial, you learned how to:
1. **Load PDF documents** using Spark's PDF reader (automatically provides document column)
2. **Build a unified ML pipeline** combining TextSplitter and TextEmbeddings stages
3. **Process documents end-to-end** - from PDF to semantic chunks to vector embeddings
4. **Analyze results** to understand embedding quality and chunk characteristics

The unified pipeline approach is efficient and follows PySpark ML patterns, making it easy to add additional stages or customize for production use.

### Key Takeaways

- **Single Pipeline**: Combines multiple transformers into one coherent workflow
- **Efficient Processing**: Data flows through stages sequentially without intermediate storage
- **Flexible Configuration**: Easy to adjust chunk sizes, embedding models, and batch sizes
- **Scalable**: PipelineModel works seamlessly with distributed Spark DataFrames

### Resources

- [ScaleDP Documentation](https://scaledp.stabrise.com/)
- [TextSplitter API Reference](https://scaledp.stabrise.com/en/latest/models/splitters/text_splitter.html)
- [TextEmbeddings API Reference](https://scaledp.stabrise.com/en/latest/models/embeddings/text_embeddings.html)
- [PySpark ML Pipelines](https://spark.apache.org/docs/latest/ml-pipeline.html)
- [Sentence-Transformers Models](https://huggingface.co/models?pipeline_tag=sentence-similarity)
- [GitHub Repository](https://github.com/stabrise/scaledp/)